# Finviz Intraday Patterns — v3

Enfocado en los dos patrones actionables identificados en v2:

1. **Duración intraday > 60 min** como filtro de calidad para Top Gainers
2. **Persistencia cross-día** — ¿qué fracción de Top Gainers repiten al día siguiente y en qué categoría acaban?

Notas metodológicas:
- `change_pct` Finviz = cambio acumulado desde el cierre anterior, NO entre snapshots. No usarlo como proxy predictivo intraday.
- Los shifts usan siempre el siguiente **día de trading** real (no +1 día calendario).
- El volumen se parsea con sufijos M/K/B.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

DB_PROD = Path.home() / 'Library/Application Support/finviz-dashboard/finviz_snapshots.db'

## 1. Carga y limpieza

In [ ]:
def parse_volume(v):
    v = str(v).strip().replace(',', '')
    if v.endswith('M'): return float(v[:-1]) * 1_000_000
    if v.endswith('K'): return float(v[:-1]) * 1_000
    if v.endswith('B'): return float(v[:-1]) * 1_000_000_000
    return pd.to_numeric(v, errors='coerce')

conn = sqlite3.connect(DB_PROD)
df = pd.read_sql("SELECT timestamp, category, ticker, price, change_pct, volume FROM snapshots", conn)
conn.close()

df['timestamp']  = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert('America/New_York')
df['date']       = df['timestamp'].dt.date
df['change_pct'] = pd.to_numeric(df['change_pct'].astype(str).str.replace('%', '', regex=False), errors='coerce')
df['price']      = pd.to_numeric(df['price'], errors='coerce')
df['volume']     = df['volume'].apply(parse_volume)
df = df.dropna(subset=['change_pct', 'volume']).copy()
df = df.sort_values(['ticker', 'date', 'timestamp']).reset_index(drop=True)

# Calendario de dias de trading reales
trading_days = sorted(df['date'].unique())
day_to_next  = {d: trading_days[i+1] for i, d in enumerate(trading_days[:-1])}

print(f"Rows: {len(df):,}  |  Días: {len(trading_days)}  |  Tickers: {df['ticker'].nunique()}")
print(f"Rango: {trading_days[0]} → {trading_days[-1]}")

## 2. Patrón 1 — Duración intraday como filtro de calidad

**Hipótesis**: un ticker que permanece en Top Gainers durante más tiempo tiene momentum más real que uno que aparece y desaparece rápido.

In [ ]:
tg = df[df['category'] == 'Top Gainers'].copy()

span = tg.groupby(['ticker', 'date']).agg(
    first_seen  = ('timestamp', 'min'),
    last_seen   = ('timestamp', 'max'),
    n_snaps     = ('timestamp', 'count'),
    change_mean = ('change_pct', 'mean'),
    change_max  = ('change_pct', 'max'),
    vol_mean    = ('volume', 'mean'),
).reset_index()

span['span_min']    = (span['last_seen'] - span['first_seen']).dt.total_seconds() / 60
span['entry_hour']  = span['first_seen'].dt.hour + span['first_seen'].dt.minute / 60

print(f"Apariciones únicas ticker-día en Top Gainers: {len(span)}")
print()
print(span['span_min'].describe().round(1))

In [ ]:
# Distribución de duraciones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(span['span_min'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(60,  color='orange', linewidth=2, linestyle='--', label='60 min')
axes[0].axvline(180, color='red',    linewidth=2, linestyle='--', label='180 min')
axes[0].set_xlabel('Duración en Top Gainers (min)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de duración intraday')
axes[0].legend()

# change_pct medio por tramo de duración
bins   = [0, 15, 60, 120, 240, 400]
labels = ['<15', '15-60', '60-120', '120-240', '>240']
span['dur_bucket'] = pd.cut(span['span_min'], bins=bins, labels=labels)
bucket_stats = span.groupby('dur_bucket', observed=True)['change_mean'].agg(['mean', 'median', 'count'])

axes[1].bar(bucket_stats.index.astype(str), bucket_stats['mean'], color='steelblue', edgecolor='white')
axes[1].set_xlabel('Duración (min)')
axes[1].set_ylabel('change_pct medio (%)')
axes[1].set_title('change_pct medio por duración en Top Gainers')
for i, (idx, row) in enumerate(bucket_stats.iterrows()):
    axes[1].text(i, row['mean'] + 1, f"n={int(row['count'])}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(bucket_stats.round(1))

In [ ]:
# Scatter: duración vs change_pct
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(span['span_min'], span['change_mean'], alpha=0.3, s=15, color='steelblue')
ax.axvline(60,  color='orange', linestyle='--', linewidth=1.5, label='60 min')
ax.set_xlabel('Duración en Top Gainers (min)')
ax.set_ylabel('change_pct medio del día (%)')
ax.set_title('Duración vs fuerza del movimiento')
ax.legend()
plt.tight_layout()
plt.show()

corr = span[['span_min', 'change_mean']].corr().iloc[0, 1]
print(f"Correlación duración vs change_pct: {corr:.3f}")

In [ ]:
# ¿A qué hora del día entran los TG que duran más?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

long_tg  = span[span['span_min'] >= 60]
short_tg = span[span['span_min'] <  60]

axes[0].hist(short_tg['entry_hour'], bins=26, alpha=0.6, label=f'<60 min (n={len(short_tg)})', color='tomato')
axes[0].hist(long_tg['entry_hour'],  bins=26, alpha=0.6, label=f'≥60 min (n={len(long_tg)})',  color='steelblue')
axes[0].set_xlabel('Hora de primera aparición (ET)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Hora de entrada por duración')
axes[0].legend()

# Pct de long TG por hora de entrada
span['entry_bin'] = (span['entry_hour'] * 2).astype(int) / 2  # bins de 30 min
pct_long = span.groupby('entry_bin').apply(
    lambda x: (x['span_min'] >= 60).mean(), include_groups=False
).reset_index(name='pct_long')

axes[1].bar(pct_long['entry_bin'], pct_long['pct_long'], width=0.4, color='steelblue')
axes[1].set_xlabel('Hora de primera aparición (ET)')
axes[1].set_ylabel('% que dura ≥ 60 min')
axes[1].set_title('¿A qué hora es más probable que el momentum sea sostenido?')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.show()

In [ ]:
# ¿A qué categoría van los TG que NO aguantan?
tg_last = df[df['category'] == 'Top Gainers'].groupby(['ticker', 'date'])['timestamp'].max().reset_index(name='tg_last')
merged  = df.merge(tg_last, on=['ticker', 'date'])
after   = merged[(merged['timestamp'] > merged['tg_last']) & (merged['category'] != 'Top Gainers')]

next_cat = after.groupby(['ticker', 'date'])['category'].first().value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
next_cat.head(10).sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Primera categoría tras salir de Top Gainers (mismo día)')
ax.set_xlabel('Frecuencia')
plt.tight_layout()
plt.show()

print(next_cat.head(10))
print(f"\nTotal que pasan a Top Losers: {next_cat.get('Top Losers', 0)} ({next_cat.get('Top Losers',0)/next_cat.sum()*100:.0f}%)")

## 3. Patrón 2 — Persistencia cross-día

**Hipótesis**: los tickers que aparecen en Top Gainers durante varios días consecutivos son candidatos PBT1 más fiables que los de una sola aparición.

In [ ]:
# Categoría dominante de cada ticker cada día
dominant = (
    df.groupby(['ticker', 'date'])['category']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index(name='cat_today')
)

# Cruzar con siguiente dia de trading
rows = []
for _, row in dominant.iterrows():
    nxt = day_to_next.get(row['date'])
    if nxt is None:
        continue
    match = dominant[(dominant['ticker'] == row['ticker']) & (dominant['date'] == nxt)]
    if len(match):
        rows.append({
            'ticker': row['ticker'],
            'date': row['date'],
            'cat_today': row['cat_today'],
            'cat_tomorrow': match.iloc[0]['cat_today']
        })

cross = pd.DataFrame(rows)
print(f"Pares ticker-día con observación al siguiente día de trading: {len(cross)}")

In [ ]:
# Matriz de transición cross-día
trans = pd.crosstab(cross['cat_today'], cross['cat_tomorrow'])
probs = trans.div(trans.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(13, 9))
sns.heatmap(probs, cmap='viridis', annot=True, fmt='.2f', linewidths=0.3, ax=ax)
ax.set_title('Probabilidad de transición cross-día (siguiente día de trading)')
ax.set_xlabel('Categoría mañana')
ax.set_ylabel('Categoría hoy')
plt.tight_layout()
plt.show()

In [ ]:
# Persistencia diagonal por categoría
diag = pd.Series({
    c: probs.loc[c, c] if c in probs.columns and c in probs.index else np.nan
    for c in probs.index
}).sort_values(ascending=False).dropna()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if v >= 0.5 else 'tomato' for v in diag.values]
ax.barh(diag.index, diag.values, color=colors)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('P(misma categoría al día siguiente)')
ax.set_title('Persistencia cross-día por categoría')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

print(diag.round(3))

In [ ]:
# Evolución diaria: ¿cuántos TG de hoy vuelven mañana?
tg_by_day = df[df['category'] == 'Top Gainers'].groupby('date')['ticker'].apply(set)

daily_stats = []
for day in trading_days[:-1]:
    nxt = day_to_next[day]
    if day not in tg_by_day.index or nxt not in tg_by_day.index:
        continue
    today_tg  = tg_by_day[day]
    tomorrow_tg = tg_by_day.get(nxt, set())
    tomorrow_any = set(df[df['date'] == nxt]['ticker'].unique())
    daily_stats.append({
        'date': day,
        'n_tg': len(today_tg),
        'n_repeat_tg': len(today_tg & tomorrow_tg),
        'n_appear_any': len(today_tg & tomorrow_any),
        'pct_repeat_tg': len(today_tg & tomorrow_tg) / len(today_tg) if today_tg else 0,
    })

ds = pd.DataFrame(daily_stats)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].bar(range(len(ds)), ds['n_tg'], label='Total TG hoy', color='steelblue', alpha=0.7)
axes[0].bar(range(len(ds)), ds['n_repeat_tg'], label='Repiten TG mañana', color='orange')
axes[0].set_ylabel('Tickers')
axes[0].set_title('Top Gainers: repetición al día siguiente')
axes[0].legend()

axes[1].plot(range(len(ds)), ds['pct_repeat_tg'], marker='o', color='steelblue')
axes[1].axhline(ds['pct_repeat_tg'].mean(), color='orange', linestyle='--',
                label=f"Media: {ds['pct_repeat_tg'].mean()*100:.0f}%")
axes[1].set_xticks(range(len(ds)))
axes[1].set_xticklabels([str(d) for d in ds['date']], rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('% que repite TG')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Media diaria de repetición en TG: {ds['pct_repeat_tg'].mean()*100:.1f}%")
print(f"Min: {ds['pct_repeat_tg'].min()*100:.0f}%  Max: {ds['pct_repeat_tg'].max()*100:.0f}%")

In [ ]:
# ¿Los TG que repiten al día siguiente son los que duraron más?
# Cruzar span con cross-día
span_indexed = span.set_index(['ticker', 'date'])
tg_cross = cross[cross['cat_today'] == 'Top Gainers'].copy()
tg_cross['persists_tg'] = tg_cross['cat_tomorrow'] == 'Top Gainers'

tg_cross = tg_cross.join(span_indexed[['span_min', 'change_mean', 'n_snaps']], on=['ticker', 'date'])
tg_cross = tg_cross.dropna(subset=['span_min'])

persist   = tg_cross[tg_cross['persists_tg']]['span_min']
no_persist = tg_cross[~tg_cross['persists_tg']]['span_min']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(no_persist, bins=30, alpha=0.6, label=f'No repite TG (n={len(no_persist)})', color='tomato')
axes[0].hist(persist,    bins=30, alpha=0.6, label=f'Repite TG (n={len(persist)})',    color='steelblue')
axes[0].set_xlabel('Duración intraday (min)')
axes[0].set_title('Duración intraday: ¿predice si repite mañana?')
axes[0].legend()

tg_cross.boxplot(column='span_min', by='persists_tg', ax=axes[1])
axes[1].set_title('Duración vs persistencia cross-día')
axes[1].set_xlabel('Repite en Top Gainers al día siguiente')
axes[1].set_ylabel('Duración intraday (min)')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f"Duración media — repite:     {persist.mean():.0f} min")
print(f"Duración media — no repite:  {no_persist.mean():.0f} min")

from scipy import stats
t, p = stats.mannwhitneyu(persist, no_persist, alternative='greater')
print(f"Mann-Whitney p-value (repite > no-repite en duración): {p:.4f}  {'*** sig' if p < 0.05 else '(no sig)'}")

In [ ]:
# ¿change_pct predice mejor la persistencia?
persist_chg    = tg_cross[tg_cross['persists_tg']]['change_mean']
nopersist_chg  = tg_cross[~tg_cross['persists_tg']]['change_mean']

print(f"change_pct medio — repite:     {persist_chg.mean():.1f}%")
print(f"change_pct medio — no repite:  {nopersist_chg.mean():.1f}%")

t2, p2 = stats.mannwhitneyu(persist_chg, nopersist_chg, alternative='greater')
print(f"Mann-Whitney p-value (change_pct): {p2:.4f}  {'*** sig' if p2 < 0.05 else '(no sig)'}")

# Threshold de change_pct para maximizar precision de prediccion de persistencia
thresholds = np.arange(10, 100, 5)
results = []
for thr in thresholds:
    above = tg_cross[tg_cross['change_mean'] >= thr]
    if len(above) < 10:
        continue
    precision = above['persists_tg'].mean()
    recall    = above['persists_tg'].sum() / tg_cross['persists_tg'].sum()
    results.append({'threshold': thr, 'n': len(above), 'precision': precision, 'recall': recall})

res_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(res_df['threshold'], res_df['precision'], marker='o', label='Precision (repite dado que >= thr)')
ax.plot(res_df['threshold'], res_df['recall'],    marker='s', linestyle='--', label='Recall')
ax.set_xlabel('Threshold de change_pct (%)')
ax.set_ylabel('Rate')
ax.set_title('Precision vs Recall: predecir repetición en TG usando change_pct')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend()
plt.tight_layout()
plt.show()

print(res_df.round(3).to_string(index=False))

## 4. Combinación de ambos patrones

¿Los TG con **duración ≥ 60 min** Y **change_pct alto** tienen mayor tasa de repetición cross-día?

## 3.5. Walk-forward validation — ¿los patrones sobreviven out-of-sample?

Con 25 días de datos usamos **expanding window**:
- Train: días 1..N (mínimo 10 días para tener muestra suficiente)
- Test: día N+1
- En cada fold derivamos el threshold óptimo en train y lo evaluamos en test
- Si el patrón es real, la precisión en test debe ser consistentemente > baseline

**Baseline**: tasa de persistencia global de Top Gainers sin ningún filtro.

In [ ]:
def build_tg_cross(span_df, dominant_df, day_map):
    """Cruza span intraday con persistencia cross-día para un subconjunto de días."""
    rows = []
    for _, row in dominant_df.iterrows():
        nxt = day_map.get(row['date'])
        if nxt is None:
            continue
        match = dominant_df[(dominant_df['ticker'] == row['ticker']) & (dominant_df['date'] == nxt)]
        if len(match):
            rows.append({
                'ticker':      row['ticker'],
                'date':        row['date'],
                'cat_today':   row['cat_today'],
                'cat_tomorrow': match.iloc[0]['cat_today'],
            })
    if not rows:
        return pd.DataFrame()
    out = pd.DataFrame(rows)
    out = out[out['cat_today'] == 'Top Gainers'].copy()
    out['persists_tg'] = out['cat_tomorrow'] == 'Top Gainers'
    out = out.join(span_df.set_index(['ticker', 'date'])[['span_min', 'change_mean']], on=['ticker', 'date'])
    return out.dropna(subset=['span_min', 'change_mean'])


def best_threshold(fold_df, feature, thresholds):
    """Devuelve el threshold que maximiza precisión en el fold de train."""
    best_thr, best_prec = thresholds[0], 0.0
    for thr in thresholds:
        sub = fold_df[fold_df[feature] >= thr]
        if len(sub) < 5:
            continue
        prec = sub['persists_tg'].mean()
        if prec > best_prec:
            best_prec = prec
            best_thr = thr
    return best_thr


MIN_TRAIN_DAYS = 10
DUR_THRESHOLDS = list(range(0, 300, 15))   # 0..285 min
CHG_THRESHOLDS = list(range(5, 100, 5))    # 5..95 %

wf_results = []

for i, test_day in enumerate(trading_days[MIN_TRAIN_DAYS:], start=MIN_TRAIN_DAYS):
    train_days = trading_days[:i]
    test_days  = [test_day]

    train_span = span[span['date'].isin(train_days)]
    test_span  = span[span['date'].isin(test_days)]

    train_dom = dominant[dominant['date'].isin(train_days + [test_day])]  # incluye test para cruzar
    test_dom  = dominant[dominant['date'].isin(test_days)]

    train_cross = build_tg_cross(span, dominant[dominant['date'].isin(train_days + [test_day])], day_to_next)
    train_cross = train_cross[train_cross['date'].isin(train_days)]

    test_cross  = build_tg_cross(span, dominant, day_to_next)
    test_cross  = test_cross[test_cross['date'].isin(test_days)]

    if len(train_cross) < 10 or len(test_cross) == 0:
        continue

    baseline_test = test_cross['persists_tg'].mean()

    # Threshold óptimo de duración en train → evaluar en test
    dur_thr  = best_threshold(train_cross, 'span_min', DUR_THRESHOLDS)
    test_dur = test_cross[test_cross['span_min'] >= dur_thr]
    prec_dur = test_dur['persists_tg'].mean() if len(test_dur) >= 3 else np.nan

    # Threshold óptimo de change_pct en train → evaluar en test
    chg_thr  = best_threshold(train_cross, 'change_mean', CHG_THRESHOLDS)
    test_chg = test_cross[test_cross['change_mean'] >= chg_thr]
    prec_chg = test_chg['persists_tg'].mean() if len(test_chg) >= 3 else np.nan

    # Combinación ambos thresholds
    test_both = test_cross[(test_cross['span_min'] >= dur_thr) & (test_cross['change_mean'] >= chg_thr)]
    prec_both = test_both['persists_tg'].mean() if len(test_both) >= 3 else np.nan

    wf_results.append({
        'test_day':    test_day,
        'n_train':     len(train_cross),
        'n_test':      len(test_cross),
        'baseline':    baseline_test,
        'dur_thr':     dur_thr,
        'prec_dur':    prec_dur,
        'chg_thr':     chg_thr,
        'prec_chg':    prec_chg,
        'prec_both':   prec_both,
        'n_both':      len(test_both),
    })

wf = pd.DataFrame(wf_results)
print(f"Folds walk-forward: {len(wf)}")
print(wf[['test_day','n_train','n_test','baseline','dur_thr','prec_dur','chg_thr','prec_chg','prec_both','n_both']].to_string(index=False))

In [ ]:
# Visualización walk-forward
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

x     = range(len(wf))
xtick = [str(d) for d in wf['test_day']]

# Panel 1: precisión vs baseline por fold
axes[0].plot(x, wf['baseline'],  marker='o', color='gray',     linestyle='--', label='Baseline (sin filtro)')
axes[0].plot(x, wf['prec_dur'],  marker='s', color='steelblue',                label='Filtro duración (OOS)')
axes[0].plot(x, wf['prec_chg'],  marker='^', color='orange',                   label='Filtro change_pct (OOS)')
axes[0].plot(x, wf['prec_both'], marker='D', color='green',                    label='Filtro combinado (OOS)')
axes[0].set_ylabel('Precisión (% repite TG)')
axes[0].set_title('Walk-forward: precisión out-of-sample vs baseline')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.05)

# Panel 2: thresholds elegidos en train a lo largo del tiempo
ax2b = axes[1].twinx()
axes[1].plot(x, wf['dur_thr'], marker='o', color='steelblue', label='Threshold duración (min)')
ax2b.plot(x,   wf['chg_thr'], marker='^', color='orange',    linestyle='--', label='Threshold change_pct (%)')
axes[1].set_ylabel('Threshold duración (min)', color='steelblue')
ax2b.set_ylabel('Threshold change_pct (%)',    color='orange')
axes[1].set_title('Estabilidad de thresholds derivados en train (¿cambian mucho?)')

# Panel 3: n_test disponibles para el filtro combinado
axes[2].bar(x, wf['n_test'],  color='lightgray', label='n total TG en test')
axes[2].bar(x, wf['n_both'],  color='steelblue', label='n tras filtro combinado')
axes[2].set_ylabel('Tickers')
axes[2].set_title('Cobertura: ¿cuántos TG quedan tras los filtros?')
axes[2].set_xticks(x)
axes[2].set_xticklabels(xtick, rotation=45, ha='right', fontsize=8)
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Resumen estadístico del walk-forward
wf_valid = wf.dropna(subset=['prec_dur', 'prec_chg', 'prec_both'])

print("=== Resumen walk-forward (OOS) ===")
print(f"Folds con datos suficientes: {len(wf_valid)} de {len(wf)}")
print()

for col, label in [('baseline', 'Baseline (sin filtro)'), ('prec_dur', 'Filtro duración'),
                    ('prec_chg', 'Filtro change_pct'), ('prec_both', 'Filtro combinado')]:
    vals = wf_valid[col].dropna()
    wins = (wf_valid[col] > wf_valid['baseline']).sum()
    print(f"{label:30s}  media={vals.mean()*100:.0f}%  mediana={vals.median()*100:.0f}%  "
          f"beats_baseline={wins}/{len(wf_valid)} folds")

print()
# Estabilidad de thresholds
print("=== Estabilidad de thresholds ===")
print(f"Threshold duración: media={wf['dur_thr'].mean():.0f} min,  std={wf['dur_thr'].std():.0f}")
print(f"Threshold change_pct: media={wf['chg_thr'].mean():.0f}%,  std={wf['chg_thr'].std():.0f}")
print()

# Cobertura: ¿cuántos tickers pasan el filtro combinado?
print("=== Cobertura media del filtro combinado ===")
coverage = wf['n_both'] / wf['n_test']
print(f"% de TG que pasan ambos filtros: media={coverage.mean()*100:.0f}%  min={coverage.min()*100:.0f}%  max={coverage.max()*100:.0f}%")
print()
print("CONCLUSIÓN:")
if wf_valid['prec_both'].mean() > wf_valid['baseline'].mean() + 0.05:
    print("  El filtro combinado mejora el baseline de forma consistente OOS — hay señal real.")
elif wf_valid['prec_both'].mean() > wf_valid['baseline'].mean():
    print("  El filtro combinado mejora marginalmente el baseline OOS — señal débil, requiere más datos.")
else:
    print("  El filtro combinado NO mejora el baseline OOS — probable overfitting en train.")

In [ ]:
tg_cross['long_duration'] = tg_cross['span_min'] >= 60
tg_cross['high_change']   = tg_cross['change_mean'] >= 20
tg_cross['both']          = tg_cross['long_duration'] & tg_cross['high_change']

combos = [
    ('Todos los TG',                tg_cross),
    ('Duración ≥ 60 min',           tg_cross[tg_cross['long_duration']]),
    ('change_pct ≥ 20%',            tg_cross[tg_cross['high_change']]),
    ('Duración ≥ 60 min + chg≥20%', tg_cross[tg_cross['both']]),
]

for label, subset in combos:
    if len(subset) == 0:
        continue
    pct = subset['persists_tg'].mean()
    print(f"{label:35s} n={len(subset):3d}  → {pct*100:.0f}% repiten en TG al día siguiente")

In [ ]:
# Heatmap: duración x change_pct → tasa de persistencia
tg_cross['dur_bin'] = pd.cut(tg_cross['span_min'], bins=[0, 30, 60, 120, 240, 500],
                              labels=['0-30', '30-60', '60-120', '120-240', '>240'])
tg_cross['chg_bin'] = pd.cut(tg_cross['change_mean'], bins=[0, 10, 20, 40, 80, 200],
                              labels=['0-10%', '10-20%', '20-40%', '40-80%', '>80%'])

pivot = tg_cross.groupby(['dur_bin', 'chg_bin'], observed=True)['persists_tg'].agg(['mean', 'count']).unstack('chg_bin')
heat_mean  = pivot['mean']
heat_count = pivot['count']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(heat_mean, annot=True, fmt='.0%', cmap='RdYlGn', ax=axes[0],
            vmin=0, vmax=1, linewidths=0.3)
axes[0].set_title('Tasa de persistencia en TG (duración x change_pct)')

sns.heatmap(heat_count, annot=True, fmt='.0f', cmap='Blues', ax=axes[1], linewidths=0.3)
axes[1].set_title('N de observaciones por celda')

plt.tight_layout()
plt.show()

## 5. Resumen ejecutivo

In [ ]:
print("=" * 60)
print("RESUMEN DE PATRONES ENCONTRADOS")
print("=" * 60)

print("\n[Patrón 1] Duración intraday como filtro de calidad")
for label, lo, hi in [('<15 min', 0, 15), ('15-60 min', 15, 60), ('≥60 min', 60, 9999)]:
    sub = span[(span['span_min'] >= lo) & (span['span_min'] < hi)]
    print(f"  TG {label:10s}: n={len(sub):3d}  change_pct medio={sub['change_mean'].mean():.0f}%")

print("\n[Patrón 2] Persistencia cross-día")
print(f"  Media diaria de repetición en TG: {ds['pct_repeat_tg'].mean()*100:.0f}%")
print(f"  Rango: {ds['pct_repeat_tg'].min()*100:.0f}% – {ds['pct_repeat_tg'].max()*100:.0f}%")

print("\n[Combinación] Filtros duración + change_pct en repetición cross-día")
for label, subset in combos:
    if len(subset) == 0: continue
    print(f"  {label:35s}: {subset['persists_tg'].mean()*100:.0f}%  (n={len(subset)})")

print("\n[Nota] Unusual Volume NO es precursor de Top Gainers (87% de casos TG aparece primero)")
print("[Nota] 58%+ de TG del día acaban en Top Losers — son movers intraday, no tendencias")